# Function Masking Analysis

Analyze the effect of Hammer-style function masking on our training data.
- Compare original vs masked datasets
- Visualize masking statistics (coverage, name lengths, type distribution)
- Show before/after examples to verify masking consistency

**Data source:** Output of `src/data/mask.py` applied to the filtered 4K subset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/info290-genai-spring2026/final-project')

In [ ]:
import json
from collections import Counter

with open('data/processed/xlam_4k.json') as f:
    original = json.load(f)

with open('data/processed/xlam_4k_masked.json') as f:
    masked = json.load(f)

print(f'Original: {len(original)} examples')
print(f'Masked:   {len(masked)} examples')

## Masking Coverage

We mask 67% of examples (following Hammer 2.1). The remaining 33% keep original names
to prevent the model from only learning to handle masked inputs.

In [ ]:
# Collect all function names from both datasets
original_names = set()
for ex in original:
    for tool in ex['tools']:
        original_names.add(tool['name'])

# Count how many examples in the masked dataset have non-original function names
masked_count = 0
unmasked_count = 0

for ex in masked:
    names = {tool['name'] for tool in ex['tools']}
    if names & original_names:  # at least one original name found
        unmasked_count += 1
    else:
        masked_count += 1

total = masked_count + unmasked_count
print(f'Masked examples:   {masked_count} ({100 * masked_count / total:.1f}%)')
print(f'Unmasked examples: {unmasked_count} ({100 * unmasked_count / total:.1f}%)')

## Before / After Examples

Show side-by-side comparison of an original example and its masked version.

In [ ]:
from src.data.mask import mask_example
import random

random.seed(42)

# Pick an example with multiple tools and a parallel call
ex = next(ex for ex in original if len(ex['answers']) >= 2 and len(ex['tools']) >= 3)

masked_ex = mask_example(ex)

print('=== ORIGINAL ===')
print(f"Query: {ex['query'][:120]}...")
print(f"Tools: {[t['name'] for t in ex['tools']]}")
for ans in ex['answers']:
    print(f"  Call: {ans['name']}({list(ans['arguments'].keys())})")

print()
print('=== MASKED ===')
print(f"Query: {masked_ex['query'][:120]}...")  # query stays the same
print(f"Tools: {[t['name'] for t in masked_ex['tools']]}")
for ans in masked_ex['answers']:
    print(f"  Call: {ans['name']}({list(ans['arguments'].keys())})")

## Masking Consistency Check

Verify that masked answer function names match masked tool names (no dangling references).

In [ ]:
inconsistent = 0
for ex in masked:
    tool_names = {t['name'] for t in ex['tools']}
    for ans in ex['answers']:
        if ans['name'] not in tool_names:
            inconsistent += 1
            break

print(f'Inconsistent examples (answer name not in tool set): {inconsistent}')
print(f'All consistent: {inconsistent == 0}')

## Parameter Name Distribution

Compare parameter name lengths and character distributions between original and masked.

In [ ]:
def get_param_name_lengths(dataset):
    lengths = []
    for ex in dataset:
        for tool in ex['tools']:
            for param_name in tool['parameters']:
                lengths.append(len(param_name))
    return lengths

orig_lengths = get_param_name_lengths(original)
masked_lengths = get_param_name_lengths(masked)

print('Parameter name lengths:')
print(f'  Original - mean: {sum(orig_lengths)/len(orig_lengths):.1f}, '
      f'min: {min(orig_lengths)}, max: {max(orig_lengths)}')
print(f'  Masked   - mean: {sum(masked_lengths)/len(masked_lengths):.1f}, '
      f'min: {min(masked_lengths)}, max: {max(masked_lengths)}')

## Call Type Distribution

Verify masking preserves the call type balance from filtering.

In [ ]:
def call_type_dist(dataset):
    counts = Counter()
    for ex in dataset:
        n = len(ex['answers'])
        if n == 0:
            counts['no_call'] += 1
        elif n == 1:
            counts['single'] += 1
        else:
            counts['parallel'] += 1
    return counts

print('Original distribution:')
for k, v in call_type_dist(original).items():
    print(f'  {k}: {v} ({100*v/len(original):.1f}%)')

print('\nMasked distribution:')
for k, v in call_type_dist(masked).items():
    print(f'  {k}: {v} ({100*v/len(masked):.1f}%)')